# 03 - Modelisation et Evaluation

**Objectif** : Entraîner un modèle de classification (Random Forest) pour prédire la gravité d'un accident (binaire : Grave vs Léger) et sauvegarder le modèle pour la mise en production.

**Secteurs** :
- 1. Chargement et Préparation des données
- 2. Entraînement du Modèle
- 3. Évaluation des Performances
- 4. Sauvegarde

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

# Configuration de l'affichage
pd.set_option('display.max_columns', None)
sns.set_theme(style='whitegrid')

## 1. Chargement et Préparation des Données

In [ ]:
def load_and_merge_data(accidents_path: str, usagers_path: str) -> pd.DataFrame:
    """
    Charge les datasets accidents et usagers, et fusionne la cible 'gravité'.
    """
    # Chargement des données principales
    df = pd.read_csv(accidents_path, low_memory=False)
    
    # Chargement de la cible (gravité) depuis le fichier usagers
    csv_params = {'sep': ',', 'encoding': 'latin1', 'low_memory': False}
    df_gravite = pd.read_csv(usagers_path, usecols=['num_acc', 'grav'], **csv_params)
    
    # Nettoyage de la cible
    df_gravite['grav'] = pd.to_numeric(df_gravite['grav'], errors='coerce')
    df_gravite = df_gravite.dropna(subset=['grav'])
    
    # Création de la cible binaire : 1 si Tué (2) ou Blessé hospitalisé (3), sinon 0
    df_gravite['grave'] = df_gravite['grav'].isin([2, 3]).astype(int)
    
    # Agrégation par accident (gravité max de l'accident)
    target = df_gravite.groupby('num_acc')['grave'].max().reset_index()
    
    # Fusion
    df_merged = df.merge(target, on='num_acc', how='left')
    df_merged['grave'] = df_merged['grave'].fillna(0).astype(int)
    
    return df_merged

df = load_and_merge_data('../data/accidents_scored.csv', '../data/usagers.csv')
print(f"Distribution de la cible :\n{df['grave'].value_counts(normalize=True)}")

In [ ]:
FEATURES = ['mois', 'jour', 'lum', 'agg', 'int', 'atm', 'col', 'catr', 'circ', 'nbv', 'vosp', 'prof', 'plan', 'lartpc', 'larrout', 'surf', 'infra', 'situ', 'env1', 'vma', 'nb_usagers', 'nb_pietons', 'pct_pietons', 'heure', 'anomaly_score']

def prepare_features(df: pd.DataFrame, features: list) -> tuple:
    """
    Sélectionne les features et la cible, gère les valeurs manquantes.
    """
    X = df[features].copy()
    y = df['grave']
    
    # Conversion numérique
    X = X.apply(pd.to_numeric, errors='coerce')
    
    # Imputation simple par la médiane (pour un modèle de prod, préférer un SimpleImputer sklearn sauvegardé)
    X = X.fillna(X.median())
    
    return X, y

X, y = prepare_features(df, FEATURES)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f'Train shape: {X_train.shape}, Test shape: {X_test.shape}')

## 2. Entraînement du Modèle

In [ ]:
# Paramètres fixés après optimisation (non montrée ici)
model_rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=15,
    random_state=42,
    n_jobs=-1,
    class_weight='balanced'
)

print("Démarrage de l'entraînement...")
model_rf.fit(X_train, y_train)
print("Entraînement terminé.")

## 3. Évaluation

In [ ]:
def evaluate_model(model, X_test, y_test):
    y_pred = model.predict(X_test)
    
    print("Rapport de classification :")
    print(classification_report(y_test, y_pred))
    
    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Léger', 'Grave'])
    
    plt.figure(figsize=(8, 6))
    disp.plot(cmap='Blues', values_format='d', ax=plt.gca())
    plt.title('Matrice de Confusion')
    plt.grid(False)
    plt.show()

evaluate_model(model_rf, X_test, y_test)

In [ ]:
def plot_feature_importance(model, feature_names):
    importances = pd.DataFrame({
        'feature': feature_names,
        'importance': model.feature_importances_
    }).sort_values('importance', ascending=False)
    
    plt.figure(figsize=(10, 8))
    sns.barplot(data=importances.head(20), x='importance', y='feature', palette='viridis')
    plt.title('Top 20 - Importance des variables')
    plt.show()
    return importances

imps = plot_feature_importance(model_rf, X_train.columns)

## 4. Sauvegarde

In [ ]:
OUTPUT_MODEL_PATH = '../model/modele_gravite_rf.joblib'
OUTPUT_COLS_PATH = '../model/colonnes_modele.joblib'

joblib.dump(model_rf, OUTPUT_MODEL_PATH)
joblib.dump(FEATURES, OUTPUT_COLS_PATH)

print(f'Modèle sauvegardé dans : {OUTPUT_MODEL_PATH}')
print(f'Colonnes sauvegardées dans : {OUTPUT_COLS_PATH}')